<a href="https://colab.research.google.com/github/pavithra08188/PRODIGY_ML_01/blob/main/Fake_news_predector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
fake = pd.read_csv("/content/drive/MyDrive/Fake[1].csv")
real = pd.read_csv("/content/drive/MyDrive/True.csv")

In [ ]:
fake["label"] = 0      # Fake News
real["label"] = 1      # Real News

df = pd.concat([fake, real], axis=0)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(df.head())

                                               title  \
0  Ben Stein Calls Out 9th Circuit Court: Committ...   
1  Trump drops Steve Bannon from National Securit...   
2  Puerto Rico expects U.S. to lift Jones Act shi...   
3   OOPS: Trump Just Accidentally Confirmed He Le...   
4  Donald Trump heads for Scotland to reopen a go...   

                                                text       subject  \
0  21st Century Wire says Ben Stein, reputable pr...       US_News   
1  WASHINGTON (Reuters) - U.S. President Donald T...  politicsNews   
2  (Reuters) - Puerto Rico Governor Ricardo Rosse...  politicsNews   
3  On Monday, Donald Trump once again embarrassed...          News   
4  GLASGOW, Scotland (Reuters) - Most U.S. presid...  politicsNews   

                  date  label  
0    February 13, 2017      0  
1       April 5, 2017       1  
2  September 27, 2017       1  
3         May 22, 2017      0  
4       June 24, 2016       1  


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["text"] = df["text"].apply(clean_text)

In [ ]:
X = df["text"]
y = df["label"]

tfidf = TfidfVectorizer(stop_words='english', max_df=0.7)

X = tfidf.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

print("Logistic Regression Accuracy:",
      accuracy_score(y_test, lr_pred))

Logistic Regression Accuracy: 0.9814031180400891


In [ ]:
nb = MultinomialNB()

nb.fit(X_train, y_train)

nb_pred = nb.predict(X_test)

print("Naive Bayes Accuracy:",
      accuracy_score(y_test, nb_pred))

Naive Bayes Accuracy: 0.9342984409799554


In [ ]:
print("\nLogistic Regression Report")
print(classification_report(y_test, lr_pred))

print("\nNaive Bayes Report")
print(classification_report(y_test, nb_pred))


Logistic Regression Report
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      4710
           1       0.98      0.98      0.98      4270

    accuracy                           0.98      8980
   macro avg       0.98      0.98      0.98      8980
weighted avg       0.98      0.98      0.98      8980


Naive Bayes Report
              precision    recall  f1-score   support

           0       0.94      0.93      0.94      4710
           1       0.93      0.94      0.93      4270

    accuracy                           0.93      8980
   macro avg       0.93      0.93      0.93      8980
weighted avg       0.93      0.93      0.93      8980



In [ ]:
news = input("Enter News: ")

news = clean_text(news)

news_vector = tfidf.transform([news])

prediction = lr.predict(news_vector)

probability = lr.predict_proba(news_vector)

if prediction[0] == 1:
    print("\nPrediction : REAL NEWS")
else:
    print("\nPrediction : FAKE NEWS")

print("Probability Score:")

print("Fake : {:.2f}%".format(probability[0][0]*100))
print("Real : {:.2f}%".format(probability[0][1]*100))

Enter News: the governmenct has annonced a education policy

Prediction : FAKE NEWS
Probability Score:
Fake : 81.48%
Real : 18.52%
